# Assignment 3 - Application of Clustering

In this assignment, we are tasked with predicting whether a given transaction is fraudulent or not. We are given a dataset on pre-processed credit card transactions and we have to train a Logistic Regression model on it for the same. The catch here is, the dataset, similar to the real world, is highly imbalanced i.e. the number of fraud transactions are very few in the same. We must figure out on how to build a model which can catch this problem despite this issue.

## Imports

Only for Colab

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !cp /content/drive/MyDrive/kaggle.json /content/kaggle.json
# !mkdir -p /root/.config/kaggle
# !cp kaggle.json /root/.config/kaggle
# !chmod 600 /root/.config/kaggle/kaggle.json

In [ ]:
import numpy as np
import pandas as pd
import json

import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub
import kaggle
from kagglehub import KaggleDatasetAdapter

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *
from sklearn.cluster import *
from sklearn.kernel_approximation import Nystroem
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.model_selection import train_test_split, GridSearchCV

import imblearn
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import *
from imblearn.under_sampling import *
from imblearn.metrics import *

## PART A : Data Exploration

Going through the Kaggle Data Information page, we note the following:
1. The dataset is already processed into numeric values to hide sensitive information (hence no need for encoding).
2. PCA has already been performed and we are given some of its principal components.
3. Two additional features are provided -
a. Time: Time elapsed in seconds since the first transaction
b. Amount: Monetary amount
4. Class contains the binary target labels

Also, as the dataset size is too big, you would have to download from the Kaggle website with your API key

In [ ]:
kaggle.api.dataset_download_files('mlg-ulb/creditcardfraud', path='./data', unzip=True)

In [ ]:
data = pd.read_csv('data/creditcard.csv')  # load the dataset

In [ ]:
data    # display some of its rows

In [ ]:
data.describe()  # some statistics on it

To see the class imbalance:

In [ ]:
neg_class = data[data['Class'] == 0]
pos_class = data[data['Class'] == 1]
neg_class_count, pos_class_count = data['Class'].value_counts()
print(f"Percentage of Positive Class data points: {len(pos_class)/ len(data) * 100}")
print(f"Percentage of Negative Class data points: {len(neg_class)/ len(data) * 100}")
print(f"Ratio of number of Negative Class to that of Positive Class : {(neg_class_count) / pos_class_count}")

In [ ]:
plt.pie(x = [neg_class_count, pos_class_count], labels=['Not Fraud', 'Fraud'], autopct='%1.1f%%')
plt.savefig('images/pie_chart_class_dis.png')
plt.show()

#### Why is this bad?
1. Most of classifiers (like Logistic Regression) is built on the assumption that the dataset is balanced.
2. Standard classification algorithms often perform poorly on such datasets, as they tend to classify most instances as the majority class to achieve high overall accuracy, ignoring the minority class.
3. The usual metrics used for gauging a model's performance, such as Accuracy, Precision, AUC-ROC and F1, are highly senstitive to the class imbalance and may indicate misleading results.

#### What can be done?
We have three options in hand. We can choose either one or more:
1. Process the Dataset
2. Use better models (which is out of scope for our problem statement)
3. Use better metrics

In [ ]:
x, y = data.drop(columns=['Class']), data['Class']

## Pipeline

In [1]:
def report_generator(y_pred, y_test, name):
    """
        Generate a comprehensive report based on various metrics
        Implemented from sklearn
    """

    x = ["Fraud", "Not Fraud"]
    actual_fraud, predict_fraud = y_test[np.where(y_test == 1)[0]], y_pred[np.where(y_test == 1)[0]]
    actual_not_fraud, predict_not_fraud = y_test[np.where(y_test == 0)[0]], y_pred[np.where(y_test == 0)[0]]

    report = {
        "Fraud": {
            "precision": precision_score(actual_fraud, predict_fraud),
            "recall": recall_score(actual_fraud, predict_fraud),
            "f1": f1_score(actual_fraud, predict_fraud),
            "bal_acc": balanced_accuracy_score(actual_fraud, predict_fraud),
            "accuracy": accuracy_score(actual_fraud, predict_fraud),
            "size": len(actual_fraud)
        },
        "Not Fraud": {
            "precision": precision_score(actual_not_fraud, predict_not_fraud),
            "recall": recall_score(actual_not_fraud, predict_not_fraud),
            "f1": f1_score(actual_not_fraud, predict_not_fraud),
            "bal_acc": balanced_accuracy_score(actual_not_fraud, predict_not_fraud),
            "accuracy": accuracy_score(actual_not_fraud, predict_not_fraud),
            "size": len(actual_not_fraud)
        },
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "pr_curve": precision_recall_curve(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "roc_curve": roc_curve(y_test, y_pred),
        "likelihood_ratios": class_likelihood_ratios(y_test, y_pred),
        "bal_acc": balanced_accuracy_score(y_test, y_pred),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_pred),
        "avg_p": average_precision_score(y_test, y_pred)
    }
    fraud, not_fraud = report['Fraud'], report['Not Fraud']

    fig, axs = plt.subplots(3, 2, figsize= (10, 10), constrained_layout = True)
    k, l = 0, 0
    for i, j in zip(fraud, not_fraud):
        if i != 'size':
            y = [fraud[i], not_fraud[j]]
            axs[k, l].bar(x, height = y)
            axs[k, l].set_ylim(0, 1.1)
            axs[k, l].set_ylabel(f"{i.capitalize()} Value")
        else:
            y = [np.log(fraud[i]), np.log(not_fraud[j])]
            axs[k, l].bar(x, height = y)
            axs[k, l].set_ylabel(f"Log of {i.capitalize()} Value")
        
        for q in range(len(x)):
            axs[k, l].text(q, y[q], np.round(y[q], 2), ha='center')  # Aligning text at center

        axs[k, l].set_title(f"{i.capitalize()} Comparison")
        l += 1
        if l == 2:
            k += 1
            l = 0

    fig.suptitle(f"Comparison of Metrics between Fraud and Not Fraud Classes in {name}")
    fig.align_xlabels(axs)
    plt.subplots_adjust(wspace=0.3, hspace=0.4)
    plt.savefig(f"images/{name}_report_skelearn.png")
    plt.show()

    cm = ConfusionMatrixDisplay(confusion_matrix=report['confusion_matrix'], display_labels=x[::-1])
    cm.plot()
    plt.savefig(f"images/Confusion_mat_{name}.png")
    plt.show()

    pr = PrecisionRecallDisplay.from_predictions(y_test, y_pred)
    pr.plot()
    plt.savefig(f"images/PR_Curve_{name}.png")
    plt.show()

    roc = RocCurveDisplay.from_predictions(y_test, y_pred)
    roc.plot()
    plt.savefig(f"images/ROC_Curve_{name}.png")
    plt.show()

    return report

In [2]:
def imbalanced_report_generator(y_pred, y_test, name):
    """
        Generate a comprehensive report based on various metrics
        Implemented from imblearn
    """

    report = classification_report_imbalanced(y_test, y_pred,
    labels=[0, 1], target_names=['Not Fraud', 'Fraud'], output_dict=True)
    d = {
        "pre": "Precison", "rec": "Recall",
        "spe": "Specificity", "f1": "F1 Score",
        "geo": "FM Index", "iba": "IBA",
        "sup": "Support", "avg_pre": "Average Precision",
        "avg_rec": "Average Recall", "avg_spe": "Average Specificity",
        "avg_f1": "Average F1 Score", "avg_geo": "Average FM Index",
        "avg_iba": "Average IBA", "total_support": "Total Support"
    }
    better_report = {
        "Fraud": dict(), "Not Fraud": dict(), 
        "Average Precision": 0,"Average Recall": 0,
        "Average Specificity": 0,
        "Average F1 Score": 0,
        "Average FM Index": 0,
        "Average Index Balanced Accuracy": 0,
        "Total Support": 0
    }
    for i in report:
        if i == "Fraud":
            k = report[i]
            for j in k:
                better_report["Fraud"][d[j]] = k[j]
        elif i == "Not Fraud":
            k = report[i]
            for j in k:
                better_report["Not Fraud"][d[j]] = k[j]
        else:
            better_report[d[i]] = report[i]
    fraud, not_fraud = better_report['Fraud'], better_report['Not Fraud']

    fig, axs = plt.subplots(4, 2, figsize= (10, 10), constrained_layout = True)
    k, l = 0, 0
    for i, j in zip(fraud, not_fraud):
        x = ["Fraud", "Not Fraud"]
        if i != 'Support':
            y = [fraud[i], not_fraud[j]]
            axs[k, l].bar(x, height = y)
            axs[k, l].set_ylim(0, 1.1)
            axs[k, l].set_ylabel(f"{i} Value")
        else:
            y = [np.log(fraud[i]), np.log(not_fraud[j])]
            axs[k, l].bar(x, height = y)
            axs[k, l].set_ylabel(f"Log of {i} Value")
        
        for q in range(len(x)):
            axs[k, l].text(q, y[q], np.round(y[q], 2), ha='center')  # Aligning text at center

        axs[k, l].set_title(f"{i.capitalize()} Comparison")
        l += 1
        if l == 2:
            k += 1
            l = 0

    axs[k, l].bar(x=["Fraud", "Not Fraud"], height=[np.log(len(pos_class)), np.log(len(neg_class))])
    axs[k, l].set_title("Sample Size Comparison")
    axs[k, l].set_xlabel(f"Class")
    axs[k, l].set_ylabel(f"Log of Size Value")
    for q in range(len(x)):
        axs[k, l].text(q, y[q], np.round(y[q], 2), ha='center')  # Aligning text at center

    fig.suptitle(f"Comparison of Metrics between Fraud and Not Fraud Classes in {name}")
    fig.align_xlabels(axs)
    plt.subplots_adjust(wspace=0.3, hspace=0.4)
    plt.savefig(f"images/{name}_report.png")
    plt.show()

    return better_report

In [ ]:
def plot_resampling(X, y, sampler, ax, title=None):
    """
        Plot the data points with labels after resampling
    """

    X_res, y_res = sampler.fit_resample(X, y)
    ax.scatter(X_res[:, 0], X_res[:, 1], c=y_res, alpha=0.8, edgecolor="k")
    if title is None:
        title = f"Resampling with {sampler.__class__.__name__}"
    ax.set_title(title)
    sns.despine(ax=ax, offset=10)

In [ ]:
def plot_decision_function(X, y, clf, ax, title=None):
    """
        Plot the data points with labels along with Vornoi regions after resampling and classification
    """

    plot_step = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, plot_step), np.arange(y_min, y_max, plot_step)
    )

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4)
    ax.scatter(X[:, 0], X[:, 1], alpha=0.8, c=y, edgecolor="k")
    if title is not None:
        ax.set_title(title)

In [ ]:
def classify(data, name, balanced=False, sampler = None):
    """
        The main pipeline code
    """

    # First Standardize the data
    x, y = StandardScaler().fit_transform(data.drop(columns=['Class']).to_numpy()), data['Class'].to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.25)
    
    pipeline = []
    if sampler is not None:
        pipeline.append(('sampler', sampler))
    if balanced:  # if we want to test with balanced class weights
        pipeline.append(('log_reg', LogisticRegression(solver="lbfgs", class_weight='balanced')))
    else:
        pipeline.append(('log_reg', LogisticRegression(solver="lbfgs")))
    pipeline = Pipeline(pipeline)

    model = pipeline.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r1 = report_generator(y_pred, y_test, name)
    r2 = imbalanced_report_generator(y_pred, y_test, name)
    
    k = ['precision','recall','f1','bal_acc','mcc','accuracy','roc_auc','avg_p']
    new_r1 = {i: r1[i] for i in k}
    new_r1['positive_likelihood_ratio'] = r1['likelihood_ratios'][0]
    new_r1['negative_likelihood_ratio'] = r1['likelihood_ratios'][1]
    new_r1['name'] = name
    df = pd.DataFrame.from_dict(new_r1, orient='index', columns=['Score']).transpose()

    return model, r1, r2, df

In [ ]:
classifiers = dict()
samplers = dict()
sk_reports = dict()
im_reports = dict()
total_df = dict()

In [ ]:
def show_decision_functions(data):
    """
        Show a comprehensive plot of all decision plots and resampling result of each resampling method
    """

    clfs = classifiers
    data2 = data.sample(n=100000, axis=0)
    fig, axs = plt.subplots(nrows=len(samplers), ncols=2, figsize=(30, 30))

    # Left plot : Decision plot ; Right plot : Resampling method
    for ax, sampler, clf in zip(axs, samplers, clfs):
        name = "None" if sampler is None else sampler.__class__.__name__
        cur_X, cur_Y = data2.drop(columns=['Class'])[['V1', 'V2']].to_numpy(), data2['Class'].to_numpy()
        plot_decision_function(cur_X, cur_Y, clf, ax[0], title=f"Using {name}")
        if sampler:
            plot_resampling(cur_X, cur_Y, sampler, ax[1], title=f"{name}")

    fig.suptitle(f"Decision function of Logistic Regression")
    fig.tight_layout()
    fig.savefig('images/decision_functions_cum.png')

## PART A : Straight Classification

What if we decide to classify directly without taking in account the class imbalance?

### Unbalanced Logistic Regression

Taking equal class weights in Logistic Regression

In [ ]:
name="Vanilla-Unbalanced"
model, r1, r2, df = classify(data, name=name)
samplers[name] = None
classifiers[name] = model
sk_reports[name] = r1
im_reports[name] = r2
total_df[name] = df

In [ ]:
df

Observations:
1. Precision is high, which means the model is careful on its predictions.
2. Recall is moderate (Precision and Recall share an approximate inverse relation) which means it misses some of the Fraud points
3. Accuracy is very high (99%)

These support our original hypothesis of the data being highly imbalanced.

### Balanced Logistic Regression

Let's see what happens when we supply a balanced class weight to the classifier

In [ ]:
name = "Vanilla-balanced"
model, r1, r2, df = classify(data, name=name, balanced=True)
samplers[name] = None
classifiers[name] = model
sk_reports[name] = r1
im_reports[name] = r2
total_df[name] = df

In [ ]:
df

Some Observations:
1. Precision is very poor while Recall is better
2. F1 score is very bad
3. Accuracy has decreased, but still high

## PART B : Apply Sampling Methods

The Dataset needs to be transformed (or more appropriately, resampled) to represent a better class distribution. Resampling means we can either <b>under-sample</b> from the majority class or <b>over-sample</b> from the minority class

### Over-sampling

We can add more examples from the minority class in the dataset to improve its percentage. There are different ways of doing the same:

#### Random Over-sampling

Randomly duplicate the records in the minority class.<br>
1. **Pro**:    Simple & Fast
2. **Con**:    Highlight specific datapoints in the class rather than generalize

In [ ]:
ros = RandomOverSampler(random_state=42)
ros.set_params(shrinkage=None)
name = "RandomOverSampler"
model, r1, r2, df = classify(data, name=name, sampler=ros)
samplers[name] = ros
classifiers[name] = model
sk_reports[name] = r1
im_reports[name] = r2
total_df[name] = df

In [ ]:
df

Some Observations:
1. The model showcases High Recall and Low Precision
2. F1 Score fails

In [ ]:
ros = RandomOverSampler(random_state=42)
ros.set_params(shrinkage=0.3)
name = "RandomOverSampler(shrinkage = 0.3)"
model, r1, r2, df = classify(data, name=name, sampler=ros)
samplers[name] = (ros)
classifiers[name] = (model)
sk_reports[name] = (r1)
im_reports[name] = (r2)
total_df[name] = df

#### Synthetic Minority Oversampling Technique (SMOTE)

In this method, we add synthetic data points in the minority class keeping the features of the data roughly maintained. We do so by using the k-nearest neighbors method.

SMOTE algorithm works in 4 simple steps:
1. Choose a minority class as the input vector.
2. Find its k nearest neighbors (k_neighbors is specified as an argument in the SMOTE() function).
3. Choose one of these neighbors and place a synthetic point anywhere on the line joining the point under consideration and its chosen neighbor.
4. Repeat the steps until the data is balanced.

The minority class is over-sampled by taking each minority class sample and introducing synthetic examples along the line segments joining any/all of the k minority class nearest neighbors. Depending upon the amount of over-sampling required, neighbors from the k nearest neighbors are randomly chosen. Default implementation currently uses five nearest neighbors. For instance, if the amount of over-sampling needed is 200%, only two neighbors from the five nearest neighbors are chosen and one sample is generated in the direction of each.

Synthetic samples are generated in the following way: Take the difference between the feature vector (sample) under consideration and its nearest neighbor. Multiply this difference by a random number between 0 and 1, and add it to the feature vector under consideration. This causes the selection of a random point along the line segment between two specific features. This approach effectively forces the decision region of the minority class to become more general.

1. **Pro**: Generates new data, hence reduces overfitting
2. **Con**: Can create noise by indiscriminatively generating points

In [ ]:
# Calculate the amount of over-sampling required:
print(f"Amount of Oversampling required for the Positive Class:{neg_class_count / pos_class_count}")

We need more than five nearest neighbors to actually perform the job.

In [ ]:
smote = SMOTE(k_neighbors=60)  # TODO: try changing k_neighbors and see what happens
name = "SMOTE(k=60)"
model, r1, r2, df = classify(data, name=name, sampler=smote)
samplers[name] = smote
classifiers[name] = model
sk_reports[name] = r1
im_reports[name] = r2
total_df[name] = df

In [ ]:
df

Some Observations:
1. Precision 

Visualize the process:

In [ ]:
neg_2 = neg_class[['V1', 'V2', 'Class']]
pos_2 = pos_class[['V1', 'V2', 'Class']]

sns.scatterplot(x=pos_2['V1'], y=pos_2['V2'])
plt.show()

#### Clustering Based Oversampling (CBO) (KMeans-SMOTE)

In this method, we first cluster the data using KMeans into some clusters. We then check for clusters with high density of minority class points. This is to ensure new points are closely related to the class. We then apply SMOTE to only these clusters.<br>
By doing this, we ensure the new data points are not noisy i.e. close to any majority class, ruining the underlying representation

1. **Pro**: Avoids Noise, better generation as per data distribution
2. **Con**: Computationally Expensive and Sensitive to hyperparameters

In [ ]:
kmeans_smote = KMeansSMOTE(
        kmeans_estimator=MiniBatchKMeans(n_clusters=100, batch_size=2048, n_init='auto'),
        random_state=42
    )
name = "kMeansSMOTE(k=100)"
model, r1, r2, df = classify(data, name=name, sampler=kmeans_smote)
samplers[name] = kmeans_smote
classifiers[name] = model
sk_reports[name] = (r1)
im_reports[name] = (r2)
total_df[name] = (df)

In [ ]:
df

Some Observations:
1. Very good F1 score of 0.85 as well as Precision and Recall.
2. The PR Curve is high as well

#### Borderline SMOTE

It focuses more on the data points from the minority class on the class border

1. **Pro**: Efficient, Better for Class boundary
2. **Con**: Sensitive to Noise as it can pick up outliers

In [ ]:
bsmote = BorderlineSMOTE()
name = "BorderlineSMOTE"
model, r1, r2, df = classify(data, name=name, sampler=bsmote)
samplers[name] = (bsmote)
classifiers[name] = (model)
sk_reports[name] = (r1)
im_reports[name] = (r2)
total_df[name] = df

#### ADASYN (Adaptive Synthetic Sampling)

1. It finds "hard-to-learn" points by assigning a difficulty ratio to each minority class. This ratio calculates the no of majority class neighbors near a minority class point. More the neighbors, harder to learn.
2. It assigns weights to each minority point directly proportional to its difficulty score
3. It then generates points around existing points proportional to their weights

1. **Pro**: Forces Classifier to focus on hard examples, excusing it for soft examples
2. **Con**: It is very sensitive to outliers.

In [ ]:
adasyn = ADASYN()
name = "ADASYN"
model, r1, r2, df = classify(data, name=name, sampler=adasyn)
samplers[name] = (adasyn)
classifiers[name] = (model)
sk_reports[name] = (r1)
im_reports[name] = (r2)
total_df[name] = df

### Undersampling

We can choose to remove certain data points from the majority class. There are different ways of doing the same:

#### Random Under-sampling

Here we remove data points randomly from the dataset. The problem is we can potentially lose vital information about the data

1. **Pro**: Improves class distribution
2. **Con**: Lose vital information about the data

In [ ]:
rus = RandomUnderSampler(random_state=42, replacement=True)# fit predictor and target variable
name = "RandomUnderSampler"
model, r1, r2, df = classify(data, name=name, sampler=rus)
samplers[name] = (rus)
classifiers[name] = (model)
sk_reports[name] = (r1)
im_reports[name] = (r2)
total_df[name] = (df)

In [ ]:
df

Some observations:
1. Bad

#### Clustering based Undersampling (ClusterCentroids)

We first cluster using KMeans from the majority class and find their centroids. It then discards the rest of the data points and uses just the centroids as the new majority class.<br>
Right from the get-go, we can see why this is bad for our case: our majority class is huge; removing this big chunk of data can lead to loss of valuable information for classification

1. **Pro**: Faster training, better reduction
2. **Con**: Catastrophic loss of information in case of large datasets

In [ ]:
cc = ClusterCentroids(sampling_strategy="majority",
    estimator=MiniBatchKMeans(n_clusters=25)
)
name = "ClusterCentroids(k=100)"
model, r1, r2, df = classify(data, name, cc)
samplers[name] = (cc)
classifiers[name] = (model)
sk_reports[name] = (r1)
im_reports[name] = (r2)
total_df[name] = (df)

In [ ]:
df

Some observations:
1. As rightly observed before, the performance is <b>not good</b>. This is as good as a Random sampler or unbalanced classifier

## Explanation of Balanced Metrics

We can use or establish better methods of evaluating a model's performance where we consider more than just the predicted positive points. We want to use those tests which are mostly independent from the data distribution and rather looks into the test's discriminative power.

Let us look at some indices which can be informative to us:<br>
We follow the conventions as:
1. TP - True Positive
2. FP - False Positive
3. TN - True Negative
4. FN - False Negative
5. AP - Actual Positive
6. AN - Actual Negative

#### $Prevalance (P) = \frac{AP}{AP + AN}$
It is defined as the ratio of number of actual positive points to the population size. It depends on the data distribution, and affected by class imbalance. This is very low for our dataset ()

#### $Precision (Positive Predictive Value) (PPV) = \frac{TP}{TP + FP}$
It counts the number of actual positives out of the predicted positives. It depends on Prevalence, and hence affected heavily by class imbalance

#### $Accuracy (A) = \frac{TP + TN}{AN + AP}$
It too heavily depends on the Prevalance.

#### $Recall (R) = \frac{TP}{TP + FN}$
It counts the number of predicted positives out of actual positives. It is instrinstic property of the test, and independent of the data

#### $F1 Score (F1): \frac{2}{F1} = \frac{1}{PPV} + \frac{1}{R}$
It is the Harmonic Mean of Precision and Recall. It too depends on Prevalence, but is a good metric as it encapsulates most of the data

### AUC-ROC
It is the Area under the ROC curve. It too depends on Prevalance.

### AUC-PR
It is the Area under the Precision-Recall Curve. It is independent of Prevalence and hence better than AUC-ROC for imbalanced datasets.

#### $Balanced Accuracy (BA) = \frac{1}{2}(\frac{TP}{TP + FN} + \frac{TN}{FP + TN})$
This is a balanced version of Accuracy taking in account of both Positive and Negative Accuracies, and hence works well for imbalanced datasets

#### Matthew's Correlation Coefficent (MCC)
This is regarded as the best metric for imbalanced datasets as it takes into account various facets of the data

Use AUCPRC or MCC or balanced accuracy instead of F1, Precision and Recall

## PART C : Results

We examine and compare different methods here

In [ ]:
# Plot of Report generated using sklearn
metrics = list(sk_reports['ADASYN'].keys())
for i in ["pr_curve", "confusion_matrix", "roc_curve", "Fraud", "Not Fraud", "likelihood_ratios"]:
    metrics.remove(i)

fig, axs = plt.subplots(4, 2, figsize = (27, 15))
i, j = 0, 0
for metric in metrics:
    print(metric)
    ax = axs[i, j]
    vals = [sk_reports[report][metric] for report in sk_reports]
    ax.bar(x=list(sk_reports.keys()), height=vals)
    ax.set_xlabel("Different clustering methods")
    ax.set_ylabel(f"{metric.capitalize()} Value")
    if not all(isinstance(i, tuple) for i in vals):
        m = 1.1 if max(vals) > 0.7 else max(vals) + 0.05
        n = 0 if min(vals) < 0.3 else min(vals) - 0.05
        ax.set_ylim(n, m)
    ax.set_title(f"{metric} value across different methods")
    for q in range(len(vals)):
        ax.text(q, vals[q], np.round(vals[q], 2), ha='center')  # Aligning text at center
    j += 1
    if j == 2:
        i += 1
        j = 0

fig.suptitle("Comparison of different methods", fontsize='xx-large')
plt.savefig("images/total_sk_report.png")
# fig.align_xlabels(axs)
plt.subplots_adjust(wspace=0.2, hspace=0.2)
plt.show()

In [ ]:
# Plot of report using imblearn
metrics = list(im_reports[0].keys())
for i in ["Fraud", "Not Fraud"]:
    metrics.remove(i)

fig, axs = plt.subplots(len(metrics) // 2, 2, figsize = (20, 20))
i, j = 0, 0
for metric in metrics:
    ax = axs[i, j]
    vals = [im_reports[report][metric] for report in im_reports]
    ax.bar(x=list(im_reports.keys()), height=vals)
    ax.set_xlabel("Different clustering methods")
    ax.set_ylabel(f"{metric.capitalize()} Value")
    if not all(isinstance(i, tuple) for i in vals):
        m = 1.1 if max(vals) > 0.7 else max(vals) + 0.05
        n = 0 if min(vals) < 0.3 else min(vals) - 0.05
        ax.set_ylim(n, m)
    ax.set_title(f"{metric} value across different methods")
    for q in range(len(x)):
        ax.text(q, y[q], np.round(y[q], 2), ha='center')  # Aligning text at center

    j += 1
    if j == 2:
        i += 1
        j = 0

fig.suptitle("Comparison of different methods", fontsize='xx-large')
plt.savefig("images/total_im_report.png")
# fig.align_xlabels(axs)
plt.subplots_adjust(wspace=0.1, hspace=0.2)
plt.show()

In [ ]:
# show_decision_functions(data)

In [ ]:
comp_df = pd.concat(total_df, axis=0)

In [ ]:
comp_df

In [ ]:
comp_df.to_csv('comparison_metrics.csv', index=False)

## PART C : Conclusion

After running various experiments for the given task on different resampling method, we can safely conclude that <b>KMeans + SMOTE</b> is the best classification pipeline, achieving <?><br>
This is because this method oversamples the minority class in the clusters with their high density, ensuring that the features are preserved. SMOTE assumes that the features of the data can be combined linearly (by taking the synthetic point along a line) which is simple but fair assumption in most cases.

If we were to cluster the data according to the first two principal components, we see that the data is not linearly seperable. This means that for a minority point, its neighbors can be majority points. SMOTE picks the neighbors indiscriminately and can pick points which are in between a majority and a minority point (also known as a tomek link). This leads to poor generalization for the minority class. Clustering ensures we pick from the points which are surronded by similar points.

Hence, the company should use <b>KMeans + SMOTE<b> method.